# Motion Sensors

Your phone knows which way up it is. Turn it sideways and the screen turns with it. Shake it and some apps undo your typing. Hold it up at night and a star map shows the sky you are pointing at. All of this comes from tiny **motion sensors** inside the phone, and in this notebook you'll read them from Python.

There are two sensors at work:

- An **accelerometer** measures pushes and pulls, including the steady pull of gravity. It is how the phone knows which way is down.
- A **gyroscope** measures how fast the phone is turning.

## Before you start

**Open this notebook on a phone or tablet.** Most laptops and desktop computers have no motion sensors. Open Codetto in the phone's browser, find this notebook on the Explore page, and carry on from here.

Then run the cell below. On an iPhone or iPad an **Allow motion sensors** button appears under the cell. Tap it, then choose **Allow**. Android phones don't ask.

> **Good to know:** On a computer without sensors, the cell ends with the error *No motion sensors found*. That is your sign to switch to a phone.

In [ ]:
from codetto import motion

motion.start()

print('Motion sensors are ready.')

# Reading the Sensors

The sensors produce about 60 measurements every second. `motion.read()` gives you the latest one, a **reading** of what the phone is doing at this moment.

| In the reading | What it holds | Measured in |
|---|---|---|
| `acceleration_with_gravity` | Pushes and pulls on the phone, including gravity | m/s² |
| `acceleration` | The same, with gravity taken away | m/s² |
| `rotation_rate` | How fast the phone is turning | degrees per second |
| `orientation` | Which way the phone is facing and tilted | degrees |

Each one is a group of three numbers. A phone that is missing a sensor gives `None` for the values it can't measure.

Hold the phone still and run the cell. Then run it again while you are moving it, and compare.

In [ ]:
reading = motion.read()

print('With gravity:', reading.acceleration_with_gravity)
print('Without gravity:', reading.acceleration)
print('Rotation rate:', reading.rotation_rate)
print('Orientation:', reading.orientation)

# Gravity and the Three Axes

The three numbers in an acceleration are its `x`, `y`, and `z` parts. Each one belongs to a direction called an **axis**, and the axes are fixed to the phone itself:

- `x` points out of the **right edge**
- `y` points out of the **top edge**
- `z` points out of the **screen**, toward you

The axes stay with the phone. Turn it to landscape and `y` still points out of the same edge.

Gravity always shows up as about `9.8` along whichever axis is pointing at the sky. Lay the phone flat on a table and `z` is about `9.8`. Stand it upright and `y` is about `9.8` instead. Turn it over and the number becomes negative.

One reading shows one moment. To follow the phone as it moves, read it again and again in a loop. The cell below runs for 15 seconds. Turn the phone every way you can and watch which axis gravity moves to.

In [ ]:
import time

def facing(x, y, z):
  biggest = max(abs(x), abs(y), abs(z))
  if biggest == abs(z):
    return 'screen up' if z > 0 else 'screen down'
  if biggest == abs(y):
    return 'top edge up' if y > 0 else 'top edge down'
  return 'right edge up' if x > 0 else 'left edge up'

end = time.time() + 15

while time.time() < end:
  x, y, z = motion.read().acceleration_with_gravity
  line = f'x={x:5.1f} y={y:5.1f} z={z:5.1f}  {facing(x, y, z)}'
  print('\r' + line.ljust(44), end='')
  time.sleep(0.1)

print()
print('Finished.')

# Tilt

Because gravity always points down, the `x` and `y` numbers tell you how far the phone is **tilted**. Lower the right edge and gravity pulls along `x`. Lower the top edge and it pulls along `y`. The further you tilt, the bigger the number, so tilting works like the stick on a game controller.

The phone can also work the tilt out for you as angles. The last two numbers of `orientation` are the tilt in degrees: forward and back, then left and right.

The cell below draws a ball that rolls downhill as you tilt the phone, like a marble on a tray. It runs for 20 seconds. Try to hold the ball in the middle of the ring.

There are two details in the code. A ball rolls *toward* the low side, which is the opposite of where gravity is measured, so `x` is subtracted. And on a canvas `y` grows downward, so `y` is added.

In [ ]:
from codetto import graphics2d
import math
import time

canvas = graphics2d.canvas()
ctx = canvas.get_context('2d')
width, height = canvas.get_width(), canvas.get_height()
x, y = width / 2, height / 2

end = time.time() + 20

while time.time() < end:
  reading = motion.read()
  tilt_x, tilt_y, _ = reading.acceleration_with_gravity
  x = min(max(x - tilt_x * 2, 15), width - 15)
  y = min(max(y + tilt_y * 2, 15), height - 15)

  canvas.clear()
  ctx.stroke_style = '#888888'
  ctx.begin_path()
  ctx.arc(width / 2, height / 2, 30, 0, 2 * math.pi)
  ctx.stroke()
  ctx.fill_style = '#e94560'
  ctx.begin_path()
  ctx.arc(x, y, 15, 0, 2 * math.pi)
  ctx.fill()

  if reading.orientation:
    _, forward, sideways = reading.orientation
    ctx.fill_style = '#333333'
    ctx.font = '16px sans-serif'
    ctx.fill_text(f'forward {forward:.0f}°  sideways {sideways:.0f}°', 10, 24)

  time.sleep(1 / 30)

print('Finished.')

# Movement and Turning

Gravity is useful for tilt, but it gets in the way when you want to know whether the phone is being *moved*. That is what `acceleration` is for. It has gravity taken away, so it reads close to `0` whenever the phone is still, however it is tilted, and jumps when the phone is pushed, shaken, or stopped suddenly.

A movement can be in any direction, so the code combines the three parts into one **strength** with Pythagoras: the square root of `x² + y² + z²`.

`rotation_rate` is the gyroscope's measurement. It is `0` when the phone isn't turning, and a quick flick of the wrist can pass 300 degrees per second.

This cell is a shake counter. It runs for 10 seconds and counts each time the strength rises above the threshold. Like a tap on a controller button, a shake is counted once when it *starts*, not on every reading while it lasts.

Change the threshold to make it more or less sensitive.

In [ ]:
import math
import time

THRESHOLD = 15 #@param {type:"slider", min:5, max:30, step:1}

shakes = 0
shaking = False
fastest_turn = 0
end = time.time() + 10

print('Shake the phone!')

while time.time() < end:
  reading = motion.read()
  x, y, z = reading.acceleration or (0, 0, 0)
  strength = math.sqrt(x * x + y * y + z * z)

  if strength > THRESHOLD and not shaking:
    shakes += 1
    print('Shake', shakes)
  shaking = strength > THRESHOLD

  for turn in reading.rotation_rate or (0, 0, 0):
    fastest_turn = max(fastest_turn, abs(turn))

  time.sleep(0.02)

print(f'{shakes} shakes. Fastest turn: {fastest_turn:.0f} degrees per second.')

# Recording a Movement

A loop with `motion.read()` only sees the reading that is current each time it asks, so it misses the ones in between. To capture a whole movement, such as a wave or a jump, use `motion.record`. It collects *every* reading for a set time and returns them together as a **clip**.

```python
clip = motion.record(seconds=2, countdown=3)
```

- `seconds` is how long to record. Leave it out to record until you tap **Stop recording**.
- `countdown=3` shows "Get ready… 3, 2, 1" first, which gives you time to get the phone into position.

While it records, a small recorder appears under the cell, and the screen is kept awake.

Run the cell, then draw a big circle in the air with the phone when the countdown ends.

In [ ]:
clip = motion.record(seconds=2, countdown=3)

print('Readings:', len(clip))
print('Length:', round(clip.seconds, 2), 'seconds')
print('Rate:', round(clip.rate), 'readings per second')
print('Columns:', clip.columns)

## Seeing the Movement

`clip.to_array()` turns the clip into a NumPy array with one row per reading and one column for each name in `clip.columns`. Plot a column against the `time` column and you can *see* the movement.

The chart below shows the three parts of `acceleration_with_gravity`. A circle in the air appears as waves, with one axis rising while another falls. Record a different movement and the shape changes, which is what lets a program tell movements apart. [[F9D25507-BA02-4876-9F0C-CD35B389BC47|Matplotlib]] has more about making charts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = clip.to_array()
times = data[:, clip.columns.index('time')]

plt.figure(figsize=(7, 3))
for name in ['accel_gravity_x', 'accel_gravity_y', 'accel_gravity_z']:
  plt.plot(times, data[:, clip.columns.index(name)], label=name[-1])

plt.xlabel('Time (seconds)')
plt.ylabel('m/s²')
plt.title('My movement')
plt.legend()
plt.show()

# Saving Recordings

`clip.save_csv(path)` writes a clip to a **CSV file**, a plain table that spreadsheets and [[FEE93C52-C481-4F9D-B2D5-671EDDF33BE7|Pandas]] can open. Save it under `/attachments/` and it is kept with the notebook, where you can download it from the attachments list.

Two things make one file able to hold many clips:

- A **label** names what a clip shows. Give it when you record, with `label='shake'`.
- Saving to a file that already exists adds the new clip to the end with the next `clip_id`. Pass `append=False` to start the file again.

The cell below records two labelled clips. Keep the phone **still** for the first and **shake** it for the second. It then opens the file with Pandas and measures how much the readings varied in each clip. The shake should give a much larger number.

A file of labelled recordings like this is exactly what is needed to train a program to recognize gestures.

In [ ]:
import pandas as pd

path = '/attachments/motion.csv'

still = motion.record(seconds=2, countdown=3, label='still')
still.save_csv(path, append=False)

shake = motion.record(seconds=2, countdown=3, label='shake')
shake.save_csv(path)

table = pd.read_csv(path)
print('Rows saved:', len(table))
print()
print('How much the readings varied:')
for label, rows in table.groupby('label'):
  print(f"  {label}: {rows['accel_gravity_x'].std():.2f}")

# Where to Go Next

You can now start the sensors, take a reading, follow gravity and tilt, detect a shake, and record, plot, and save a whole movement.

The sensors switch off by themselves when you leave the notebook. To switch them off sooner, call `motion.stop()`.

Some things to try:

- Turn the tilt cell into a maze game, with walls the ball must not touch.
- Make a step counter. Walking gives the strength of `acceleration` a bump with every step.
- Play a different note for each way up the phone can be, with [[807A26DA-D106-4E6B-9921-8B6BCC8054E2|Playing Notes and Sounds]].
- Record the same movement three times and plot the clips together. How alike are they?
- Compare tilting a phone with pushing a stick in [[8962E963-B160-403F-AB64-553FE1C2FF8D|Game Controllers]].

# Check Your Understanding